Load ERA5 Wind Speed Data

In [1]:
from pathlib import Path
import numpy as np
import xarray as xr
import torch

# Locate project
project_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "experiments" / "02_climate").is_dir()
)

experiment_dir = project_root / "experiments" / "02_climate"
data_dir = project_root / "datasets" / "climate" / "raw"

era5_path = data_dir / "era5_germany_jan2020_10m_wind.nc"

# Read ERA5 data
ds = xr.open_dataset(era5_path)

u10 = ds["u10"]
v10 = ds["v10"]

# Determine time dimension
time_dim = next(
    dim for dim in u10.dims
    if dim in ("time", "valid_time")
)

# Wind speed magnitude
wind_speed = np.hypot(
    u10.isel({time_dim: slice(0, 400)}),
    v10.isel({time_dim: slice(0, 400)})
)

print("=== ERA5 WIND SPEED ===")
print("File:", era5_path.name)
print("Variables:", list(ds.data_vars))
print("U10 shape:", u10.shape)
print("V10 shape:", v10.shape)
print("Wind speed shape:", wind_speed.shape)
print("Dimensions:", wind_speed.dims)
print("Units:", u10.attrs.get("units", "unknown"))
print("Time steps:", wind_speed.sizes[time_dim])
print("Time range:",
      str(ds[time_dim].values[0]),
      "to",
      str(ds[time_dim].values[-1]))


=== ERA5 WIND SPEED ===
File: era5_germany_jan2020_10m_wind.nc
Variables: ['u10', 'v10']
U10 shape: (744, 33, 41)
V10 shape: (744, 33, 41)
Wind speed shape: (400, 33, 41)
Dimensions: ('valid_time', 'latitude', 'longitude')
Units: m s**-1
Time steps: 400
Time range: 2020-01-01T00:00:00.000000000 to 2020-01-31T23:00:00.000000000


Extract ERA5 Wind Speed at 10 German Locations

In [2]:
# 1. Select 10 representative locations in Germany
locations = {
    "Berlin":    (52.5200, 13.4050),
    "Hamburg":   (53.5511, 9.9937),
    "Munich":    (48.1351, 11.5820),
    "Cologne":   (50.9375, 6.9603),
    "Frankfurt": (50.1109, 8.6821),
    "Stuttgart": (48.7758, 9.1829),
    "Leipzig":   (51.3397, 12.3731),
    "Dresden":   (51.0504, 13.7373),
    "Hannover":  (52.3759, 9.7320),
    "Nuremberg": (49.4521, 11.0767),
}

names = list(locations.keys())

target_lats = xr.DataArray(
    [locations[name][0] for name in names],
    dims="site"
)

target_lons = xr.DataArray(
    [locations[name][1] for name in names],
    dims="site"
)

# 2. Find nearest ERA5 grid cell for each location
selected = wind_speed.sel(
    latitude=target_lats,
    longitude=target_lons,
    method="nearest"
).transpose(time_dim, "site")

wind_raw = selected.values.astype(np.float32)

actual_lats = selected["latitude"].values.tolist()
actual_lons = selected["longitude"].values.tolist()

# 3. Validate extracted data
assert wind_raw.shape == (400, 10)
assert np.isfinite(wind_raw).all()

grid_points = list(zip(actual_lats, actual_lons))
assert len(set(grid_points)) == 10, "Duplicate ERA5 grid points"

# 4. Save dataset
output_dir = experiment_dir / "data"
output_dir.mkdir(parents=True, exist_ok=True)

save_path = output_dir / "era5_wind_10sites.pt"

torch.save({
    "wind_raw": torch.from_numpy(wind_raw),
    "location_names": names,
    "target_latitudes": target_lats.values.tolist(),
    "target_longitudes": target_lons.values.tolist(),
    "grid_latitudes": actual_lats,
    "grid_longitudes": actual_lons,
    "units": "m s**-1",
    "source": era5_path.name,
}, save_path)

# 5. Summary
print("=== ERA5 WIND SPEED - 10 LOCATIONS ===")
print("Dataset shape:", wind_raw.shape)
print("Value range (m/s):",
      round(float(wind_raw.min()), 3),
      "-",
      round(float(wind_raw.max()), 3))

print("\nSelected ERA5 grid points:")
for i, name in enumerate(names):
    print(
        f"{i + 1:2d}. {name:<12} "
        f"({actual_lats[i]:.2f}, {actual_lons[i]:.2f})"
    )

print("\nSaved:", save_path)


=== ERA5 WIND SPEED - 10 LOCATIONS ===
Dataset shape: (400, 10)
Value range (m/s): 0.042 - 8.241

Selected ERA5 grid points:
 1. Berlin       (52.50, 13.50)
 2. Hamburg      (53.50, 10.00)
 3. Munich       (48.25, 11.50)
 4. Cologne      (51.00, 7.00)
 5. Frankfurt    (50.00, 8.75)
 6. Stuttgart    (48.75, 9.25)
 7. Leipzig      (51.25, 12.25)
 8. Dresden      (51.00, 13.75)
 9. Hannover     (52.50, 9.75)
10. Nuremberg    (49.50, 11.00)

Saved: c:\Users\Win 11 New\Desktop\pa gk\experiments\02_climate\data\era5_wind_10sites.pt
